# High-norm tokens, attention sinks and massive activation channels in DiTs

One analysis, run unchanged across three model families.

This notebook answers three questions, in this order:

| | Question | Where |
|---|---|---|
| **Sink layers** | In which layers do attention sinks exist, and how does that line up with high-norm tokens and massive activation channels? | §5 |
| **Shared direction** | What is the direction v\* that the register tokens share? Is it really one direction, is it prompt/seed-invariant, and is the top massive-activation channel its main carrier? | §7 |
| **Sink shape** | What do the sinks look like inside a given layer, per-head, with the sink columns called out? | §6 |

### Models

Every cell below runs unchanged on:

* `flux1-schnell`, `flux1-dev`: 19 dual + 38 single blocks
* `flux2-dev`, `flux2-klein`: 8 dual + 48 single blocks, *parallel* single-stream blocks
* `pixart-sigma-512`, `pixart-sigma-1024`: 28 single-stream blocks, text in cross-attention

> **A naming note.** There is no *FLUX.2 schnell*. In the FLUX.2 line the small, fast,
> openly-released model is **klein** (`black-forest-labs/FLUX.2-klein-base-9B`), which is
> what the alias `flux2-schnell` resolves to here. `flux2-dev` is ~32B parameters and
> needs 4-bit quantisation to fit on one 40 to 48 GB GPU (§2).

Run order: §1, §2, (§3 optional, no GPU), §4, §5 through §7. §8 compares models; §9 exports.

### Plain-language reading guide

The model turns an image into a grid of **patch tokens** and repeatedly updates one vector per
patch. This notebook follows three kinds of unusually large structure through those updates:

1. A **high-norm token** is a patch vector much longer than a typical patch vector in the same
   layer. Think of vector length as how loudly that patch is represented.
2. An **attention sink** is a patch that many other patches look at. A value of $10\times$
   uniform means it receives ten times the attention it would receive if attention were spread
   evenly. A sink need not correspond to a meaningful image object; it can act as internal
   scratch space or a register.
3. A **massive activation channel** is one hidden coordinate whose absolute value is unusually
   large. It asks whether the high-norm token is loud in many coordinates or mainly carried by a
   small number of coordinates.

#### How to read every figure

* Read the **title** for the model, layer and block. `dual` blocks mix text and image streams;
  `single` blocks operate after those streams have been joined (the exact implementation varies
  by model family).
* Read each **colour bar before comparing colours**. Yellow means “large” only relative to that
  panel's stated units. Logarithmic colour scales preserve both the ordinary bulk and rare outliers.
* A layer-distribution plot is read left-to-right through network depth. A detached bright point or
  streak above the bulk is an outlier population.
* In an attention matrix, rows are querying tokens and columns are receiving tokens. A bright
  **vertical stripe** therefore means many queries select the same key: an attention sink.
* Spatial maps reshape token ids back to patch positions. They locate an effect but do not by
  themselves prove that the corresponding image content caused it.
* Overlap/correlation figures test association, not causation. The interventions in §9 edit the
  representation and test whether the sink follows its magnitude or direction.

#### What the notebook produces

The sweep saves measured layer records and the generated images. The analysis then exports
paper-ready PDF figures, CSV tables of layer/head/token metrics, suggested LaTeX captions, and
`vstar.pt`, the fitted shared register direction. Sections 5 and 6 answer where and what; §7 explains
the shared direction; §8 checks robustness; §9 performs causal edits; §10 compares architectures;
and §11 exports reusable results.


---
## 1. Setup

In [ ]:
# 1.1: Dependencies.
# Pinned: FLUX.2 support landed in diffusers 0.36, and the analysis reads
# diffusers internals, so an unpinned upgrade is the usual way this breaks.
import importlib.util, subprocess, sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "diffusers==0.40.0", "transformers>=4.56", "accelerate", "safetensors",
                    "sentencepiece", "protobuf", "huggingface_hub"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "pandas", "matplotlib", "scipy"], check=True)
    # Only needed for 4-bit FLUX.2-dev (see 2.2).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=False)
print("colab:", IN_COLAB)

In [ ]:
# 1.2: The analysis library.
# Uses the repository this notebook sits in. Clones REPO_URL only if the root isn't found.
import sys, os
from pathlib import Path

REPO_URL = ""  # optional git remote to clone when the repository root is not found
REPO_BRANCH = "main"

def _ensure_ditsinks():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "ditsinks" / "__init__.py").exists():
            sys.path.insert(0, str(parent))
            return parent
    target = Path("sparse-structures-dit")
    if not (target / "ditsinks" / "__init__.py").exists():
        if not REPO_URL:
            raise RuntimeError("Run from the repository root, or set REPO_URL to a git remote.")
        os.system(f"git clone -q --branch {REPO_BRANCH} --depth 1 {REPO_URL} {target}")
    sys.path.insert(0, str(target.resolve()))
    return target.resolve()

REPO_ROOT = _ensure_ditsinks()
import ditsinks
print("ditsinks", ditsinks.__version__, "from", REPO_ROOT)

In [ ]:
# 1.3: Hugging Face auth and where results go.
# FLUX.1-dev and the FLUX.2 checkpoints are gated: accept the licence on the model
# page first, then log in. FLUX.1-schnell and PixArt-Sigma need no token.
if IN_COLAB:
    from huggingface_hub import notebook_login
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT_DIR = "/content/drive/MyDrive/ditsinks"
    except Exception:
        OUTPUT_DIR = "/content/ditsinks"
    notebook_login()
else:
    OUTPUT_DIR = "outputs/ditsinks"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("results ->", OUTPUT_DIR)

In [ ]:
# 1.4: Imports and the visual theme.
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

from ditsinks import SweepConfig, run_sweep, registry_table, resolve_model
from ditsinks import figures as F, metrics as M, style as st
from ditsinks.vstar import fit_vstar, attach_exact_selection

st.use("paper")          # publication defaults: serif/STIX type, closed axes, 400 dpi
                         # ("talk" is the same system at presentation sizes)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

---
## 2. Pick a model

Every figure below is built for a paper: Times-like serif type (STIX), a closed axes box with
inward ticks, quantities named on the axes rather than in a note, and 400 dpi output. Each
figure also carries a suggested LaTeX caption in `fig.dv_caption` (§9.2 prints them all).

Colour does two jobs and each has one rule. *Which phenomenon* a curve belongs to uses three
Okabe-Ito slots, fixed for the whole notebook:
**<span style="color:#0072B2">blue = high-norm tokens</span>**,
**<span style="color:#D55E00">vermillion = attention sinks</span>**,
**<span style="color:#009E73">green = massive activation channels</span>**, which clear the
colour-vision-deficiency and contrast thresholds and survive greyscale printing. *How many*
tokens fall in a bin uses `inferno` in every density panel, so the colour axis means the same
thing everywhere; attention probability, a different quantity, uses `cividis`.


In [ ]:
# 2.1: What can be run.
display(pd.DataFrame(registry_table()))

In [ ]:
# 2.2: The experiment.
# Change MODEL and re-run everything below; nothing else is model-specific.
MODEL = "flux1-schnell"          # flux1-schnell | flux1-dev | flux2-dev | flux2-klein | pixart-sigma-512

PROMPTS = [
    "A close-up portrait of an astronaut in a reflective helmet, cinematic lighting, ultra detailed",
    "A futuristic city street at night after rain, neon reflections, detailed architecture",
    "A red fox sitting in a snowy forest, shallow depth of field, cinematic",
]
SEEDS = [0, 1]
SIZE = 512                       # 512 keeps the attention statistics cheap; 1024 works too

# Layer controls for the layer-specific figures in §6.
# Use "auto" for the strongest stored sink layer, [12] for one layer, [12, 30] for several,
# or "all" to export the spatial/profile/scatter figures for every measured layer.
FIGURE_LAYERS = "auto"

# Full attention matrices are much larger than the reduced profiles used by the spatial figure.
# Use "auto" for four representative depths, or a list such as [12, 30]. "all" is supported but
# can require substantial RAM/disk because it stores O(heads * tokens^2) values per layer.
FULL_ATTENTION_LAYERS = "auto"

# Approved random-overlap reference. Keep True for the paper; set False for versions without it.
SHOW_CHANCE_BASELINE = True

spec = resolve_model(MODEL)
cfg = SweepConfig(
    model=MODEL,
    prompts=PROMPTS,
    seeds=SEEDS,
    height=SIZE, width=SIZE,
    # capture the last denoising step by default; add more to see the sinks form
    capture_steps=None,
    # layers=None  ->  sweep EVERY layer, which is what the sink-layers question needs
    layers=None,
    # Full maps are required only by the per-head attention-matrix grid.
    focus_layers=([] if FULL_ATTENTION_LAYERS == "auto" else
                  (list(range(spec.n_layers)) if FULL_ATTENTION_LAYERS == "all" else
                   list(FULL_ATTENTION_LAYERS))),
    focus_layer_fractions=([0.05, 0.35, 0.65, 0.95]
                           if FULL_ATTENTION_LAYERS == "auto" else []),
    output_dir=OUTPUT_DIR,
    # FLUX.2-dev is ~32B: switch this on to fit it on a single 40-48GB GPU.
    quantize_4bit=(MODEL == "flux2-dev"),
)

print(f"{spec.key}  ({spec.repo_id})")
print(f"  family {spec.family} | ~{spec.approx_params_b}B params | "
      f"{cfg.num_inference_steps} steps | guidance {cfg.guidance_scale} | {cfg.height}x{cfg.width}")
print(f"  capturing steps {cfg.capture_steps} for {len(PROMPTS)} prompts x {len(SEEDS)} seeds "
      f"= {len(PROMPTS) * len(SEEDS)} generations")
print("\n" + spec.notes)
if cfg.quantize_4bit:
    print("\n4-bit quantisation is ON. It makes the model fit, but it perturbs activations: "
          "treat the numbers as qualitative unless you can also run bf16.")

---
## 3. Dry run: check everything without a GPU  *(optional but cheap)*

Two independent checks, both offline:

1. **Integration check**: builds the *real* diffusers block classes for your model's family at
   toy width with random weights, runs the full capture, and asserts the attention tap does not
   change the model's output. This is what catches a diffusers version bump before it costs you
   an hour of GPU time.
2. **Figure preview**: renders every figure on *invented* data shaped like a published FLUX run,
   so you can see what the outputs look like before committing to the real thing.
   **Nothing in the preview is a measurement.**


In [ ]:
# 3.1: Integration check on real diffusers blocks (random weights, CPU, ~5 s).
from ditsinks.synthetic import build_tiny, planted_direction
from ditsinks.attention_patch import AttentionTap

fam = spec.family
tiny_cfg = SweepConfig(model=f"tiny-{fam}", prompts=["a"], seeds=[0],
                       num_inference_steps=2, capture_steps=[0, 1], height=128, width=128)
tiny_res = run_sweep(tiny_cfg, progress=False)

rec = next(iter(tiny_res.records.values()))
assert rec.incoming_img2img is not None and rec.channel_absmax is not None
assert torch.allclose(rec.incoming_img2img.sum(-1), torch.ones(rec.n_heads), atol=1e-4)

# The tap must be inert: same inputs, same output, with and without observation.
bundle = build_tiny(fam, steps=1, grid=6)
v = planted_direction(bundle.d_model)
def _run_once():
    grabbed = {}
    h = bundle.transformer.transformer_blocks[-1].register_forward_hook(
        lambda m, a, k, out: grabbed.__setitem__("o", out[1] if isinstance(out, tuple) else out),
        with_kwargs=True)
    try:
        with torch.no_grad():
            bundle.call(bundle.transformer, 0, 0, v)
    finally:
        h.remove()
    return grabbed["o"].clone()

clean = _run_once()
with AttentionTap(fam, lambda *a, **k: None):
    tapped = _run_once()
assert torch.equal(clean, tapped), "the attention tap changed the model output"

print(f"OK  family={fam}  layers={tiny_res.meta['n_layers']}  "
      f"records={len(tiny_res.records)}")
print(f"    attention observed at: {tiny_res.meta['attention_tap']}")
print("    tap is inert: model output bit-identical with and without capture")

In [ ]:
# 3.2: Figure preview on MOCK data (no measurement, shapes only).
from ditsinks.mock import make_mock_result, MOCK_BANNER
from ditsinks.vstar import fit_vstar as _fit

_mock = make_mock_result(focus_layers=(30,))
print(MOCK_BANNER)
F.fig_layer_atlas(_mock); plt.show()
F.fig_vstar_identity(_fit(_mock), _mock); plt.show()
del _mock

---
## 4. Run the sweep

One generation per (prompt, seed). At every layer of every captured denoising step the run records

* residual-stream **token norms** (`pre_block`, `attn_out`, `post_block`, and the post-attention
  residual where the architecture has one),
* the **incoming image→image attention** profile per head, plus how much mass leaks to text keys,
* per-channel **peak activation**, and the per-token values of the loudest channels,
* the full hidden vectors of the loudest tokens and of matched ordinary tokens (for §7).

Nothing of size `O(N²)` or `O(N·C)` is kept, except the per-head attention maps at the
`focus_layers`, which is what makes sweeping all 57 layers affordable.


In [ ]:
# 4.1: Run it. FLUX.1-schnell at 512x512, 6 generations: a few minutes on an A100.
result = run_sweep(cfg)
result.save()
cfg.save()

print(f"\ncaptured {len(result.records)} layer-records across {result.meta['n_layers']} layers")
print(f"attention observed at: {result.meta['attention_tap']}")
print(f"patch grid: {result.meta['grid']}  |  focus layers with full attention maps: {cfg.focus_layers}")

In [ ]:
# 4.2: (Re)load a saved sweep instead of re-running.
# from ditsinks import SweepResult
# result = SweepResult.load(Path(OUTPUT_DIR) / spec.key / "sweep_records.pt")
# cfg = result.cfg

---
## 5. Sink layers: in which layers do attention sinks exist?

The reference view is the **layer-wise distribution**: for each layer, bin the per-token (or
per-channel) values and colour each cell by how many fall in it, on a log scale. The bulk of the
distribution reads as a bright ridge; a handful of outlier tokens stay visible as isolated cells
far above it. §5.1 builds that view for each phenomenon separately, §5.2 stacks all three on one
shared layer axis, and §5.3 reduces each to a single curve so the depth ranges can be compared
numerically.

Notation used on the axes ($t$ indexes image tokens, $c$ channels, $h$ heads):

| symbol | meaning |
|---|---|
| $\|x_t\|_2$ | residual-stream $\ell_2$ norm of image token $t$ after the block |
| $a_h(t)$ | attention head $h$ places on key $t$, averaged over image queries |
| $\bar a(t)$ | $a_h(t)$ averaged over heads; sums to 1 over image keys |
| $N\,\bar a(t)$ | the same in multiples of the uniform share $1/N$ |
| $\max_t \|x_{t,c}\|$ | peak activation of channel $c$ over image tokens |


### 5.1 One phenomenon at a time

The first cell is the residual-norm view on its raw scale, the same construction as the
layer-wise norm figure this analysis started from. The second divides each layer by its own
median first, which matters because the bulk norm grows with depth: on a raw linear axis the
early layers are compressed into the baseline and only the late outliers are legible.


In [ ]:
# 5.1a: Residual-stream norms, raw scale.
fig = F.fig_layer_distribution(result, "norm",
                               path=f"{OUTPUT_DIR}/{spec.key}/figs/fig1a_norm_distribution.pdf")
plt.show()

In [ ]:
# 5.1b: Each quantity against its own layer median (norms, channels) or the
#        uniform attention share (sinks).
for q, norm_mode in [("norm", "median"), ("attention_ratio", "none"), ("channel", "median")]:
    F.fig_layer_distribution(result, q, normalize=norm_mode, log_y=True,
                             path=f"{OUTPUT_DIR}/{spec.key}/figs/fig1b_{q}.pdf")
    plt.show()

### 5.2 All three on one layer axis

The main figure for the sink-layers question.


In [ ]:
# 5.2: The layer atlas.
atlas = F.fig_layer_atlas(result, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig2_layer_atlas.pdf")
plt.show()
print(atlas.dv_caption)

### 5.3 Reduced to one curve per phenomenon

Each phenomenon becomes an **extremal-to-typical ratio**, the largest value at a layer divided by
that layer's own reference level. The three are dimensionless and therefore share one axis
without a second y-scale:

$$\frac{\max_t \|x_t\|}{\mathrm{med}_t \|x_t\|}, \qquad
N \max_t \bar a(t), \qquad
\frac{\max_c p_c}{\mathrm{med}_c\, p_c}, \quad p_c = \max_t |x_{t,c}| .$$

A value of 1 means the largest value at that layer equals its reference level, i.e. nothing is
anomalous there.


In [ ]:
# 5.3: Depth profile, with the layers above threshold marked.
F.fig_layer_summary(result, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig3_depth_profile.pdf")
plt.show()

The atlas above puts the three panels on a common footing: attention in multiples of
the uniform share, norms and channel peaks relative to each layer's own median. That is what makes
the depth ranges comparable, but it hides the values themselves. The next cell plots the same
three distributions in the units they were measured in, with the per-layer median drawn on top.

In [ ]:
# 5.2b: The same three distributions in absolute units.
F.fig_layer_atlas(result, normalize="none",
                  path=f"{OUTPUT_DIR}/{spec.key}/figs/fig2b_layer_atlas_absolute.pdf")
plt.show()

In [ ]:
# 5.4: The same statistics unpacked, plus the coupling panel: are the top-1% of
#        tokens by norm the same tokens as the top-1% by incoming attention?
F.fig_layer_panels(result, show_chance=SHOW_CHANCE_BASELINE, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig4_layer_panels.pdf")
plt.show()

### 5.5 The answer, in words and in a table

In [ ]:
# 5.5: sink layers answered.
layer_df = M.layer_table(result)
print(M.answer_where_are_the_sinks(result, layer_df))

verdict = M.sink_layer_verdict(layer_df, threshold=cfg.sink_ratio_threshold)
display(verdict[["layer", "block_name", "sink_ratio", "head_consensus", "mass_top1pct",
                 "entropy_norm", "n_highnorm", "n_massive_channels", "norm_ratio",
                 "jaccard", "jaccard_chance", "has_sinks"]].round(3))

In [ ]:
# 5.6: Sink strength for every (layer, head) pair. A solid bright column means the
#        whole block converges on one key token; scattered cells mean each head picks its own.
F.fig_layer_head_sinks(result, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig5_layer_head_sinks.pdf")
plt.show()

In [ ]:
# 5.7: The text-token confound (FLUX only; PixArt keeps text in cross-attention).
# Sink strengths above are renormalised over image keys. This figure gives the fraction of the
# raw mass those numbers describe, which bounds how much of a head an image sink can explain.
try:
    F.fig_text_vs_image_sink(result, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig6_text_vs_image.pdf")
    plt.show()
except ValueError as e:
    print("skipped:", e)

---
## 6. Sink shape: what does a sink look like inside a layer?

A sink is a **bright vertical stripe** in the attention matrix: every query row spends part of its
mass on the same key column. The grid below is one panel per head for a single layer, with

* the sink column marked and the token id in each head's title,
* the **text | image boundary** drawn, so a padding sink cannot be mistaken for an image register,
* mean pooling (not striding) on both axes, so a one-token-wide sink column survives downsampling;
  striding will miss it unless the stride happens to land on it.

Full attention maps are only stored at `cfg.focus_layers`. To inspect a different layer, set
`focus_layers=[L]` in §2 and re-run §4.


In [ ]:
# 6.1: Select one, several, or every measured layer for the layer-specific figures.
available_maps = sorted({r.layer for r in result.records.values() if r.attn_map is not None})
measured_layers = sorted({r.layer for r in result.records.values()})
ranked = verdict.set_index("layer")

if FIGURE_LAYERS == "auto":
    candidates = available_maps or measured_layers
    FIGURE_LAYER_LIST = [int(ranked.loc[candidates, "sink_ratio"].idxmax())]
elif FIGURE_LAYERS == "all":
    FIGURE_LAYER_LIST = measured_layers
else:
    FIGURE_LAYER_LIST = sorted({int(layer) for layer in FIGURE_LAYERS})
    missing = sorted(set(FIGURE_LAYER_LIST) - set(measured_layers))
    if missing:
        raise ValueError(f"FIGURE_LAYERS contains uncaptured layers: {missing}")

FOCUS = FIGURE_LAYER_LIST[0]  # retained for the v* and intervention cells below
display(ranked.loc[FIGURE_LAYER_LIST, ["block_name", "sink_ratio", "head_consensus"]].round(2))
print(f"Generating layer-specific figures for: {FIGURE_LAYER_LIST}")
print(f"Full per-head matrices are available for: {available_maps}")


In [ ]:
# 6.2: Per-head attention matrices (only layers captured in FULL_ATTENTION_LAYERS).
for layer in [layer for layer in FIGURE_LAYER_LIST if layer in available_maps]:
    F.fig_attention_head_grid(
        result, layer, prompt_id=0,
        path=f"{OUTPUT_DIR}/{spec.key}/figs/fig7_head_grid_L{layer}.pdf")
    plt.show()
if not set(FIGURE_LAYER_LIST) & set(available_maps):
    print("No selected layer has a full attention matrix. Add it to FULL_ATTENTION_LAYERS in §2 "
          "and re-run the sweep; the reduced spatial figures below still work for every layer.")


In [ ]:
# 6.3: Incoming-attention profile and high-norm map for any selected layer.
# The wide constrained layout keeps panel titles, axis labels, and colour bars separated.
for layer in FIGURE_LAYER_LIST:
    F.fig_sink_profile(
        result, layer, prompt_id=0,
        path=f"{OUTPUT_DIR}/{spec.key}/figs/fig8_sink_profile_L{layer}.pdf")
    plt.show()


In [ ]:
# 6.4: Threshold-free flagship spatial figure.
# Attention uses viridis and post-block norm uses magma, both in raw measured units.
# No arbitrary top-k selection is shown in this descriptive figure.
for layer in FIGURE_LAYER_LIST:
    fig = F.fig_spatial_token_maps(
        result, layer, prompt_id=0,
        path=f"{OUTPUT_DIR}/{spec.key}/figs/fig12_spatial_maps_L{layer}.pdf")
    plt.show()
    print(fig.dv_caption)


In [ ]:
# 6.4b: Top-k overlap is a separate robustness figure, not part of the flagship map.
# The four panels use the exact categorical palette of the original figure.
OVERLAP_TOP_K = (1, 3, 5, 10)
for layer in FIGURE_LAYER_LIST:
    fig = F.fig_spatial_overlap_maps(
        result, layer, prompt_id=0, ks=OVERLAP_TOP_K,
        path=f"{OUTPUT_DIR}/{spec.key}/figs/fig12b_spatial_overlap_L{layer}.pdf")
    plt.show()
    print(fig.dv_caption)


In [ ]:
# 6.5: Rank agreement across the whole distribution, not just the extreme tail.
for layer in FIGURE_LAYER_LIST:
    F.fig_rank_rank(result, layer,
                    path=f"{OUTPUT_DIR}/{spec.key}/figs/fig13_rank_rank_L{layer}.pdf")
    plt.show()


In [ ]:
# 6.6: Per-token profiles and norm/attention scatter for one head, at selected layers.
HEAD = 0
for layer in FIGURE_LAYER_LIST:
    F.fig_norm_attention_curves(
        result, layer, head=HEAD,
        path=f"{OUTPUT_DIR}/{spec.key}/figs/fig14_token_profiles_L{layer}.pdf")
    plt.show()
    F.fig_norm_vs_attention_scatter(
        result, layer, head=HEAD,
        path=f"{OUTPUT_DIR}/{spec.key}/figs/fig15_norm_vs_attention_L{layer}.pdf")
    plt.show()


In [ ]:
# 6.7: Compare two depths side by side: a layer before the sinks appear, and one after.
first_sink = int(verdict[verdict["has_sinks"]]["layer"].min()) if verdict["has_sinks"].any() else None
if first_sink is None:
    print(f"No layer crosses the sink threshold ({cfg.sink_ratio_threshold}x uniform) in this run, "
          f"so there is no 'before' to contrast. The strongest layer is L{FOCUS} at "
          f"{verdict.loc[verdict['layer'] == FOCUS, 'sink_ratio'].iloc[0]:.1f}x.")
else:
    before = max([l for l in available_maps if l < first_sink], default=None)
    if before is None:
        print("every layer with a stored map already has sinks — add an earlier fraction to "
              "focus_layer_fractions in §2 to see the 'before' picture")
    else:
        F.fig_attention_head_grid(result, before, prompt_id=0,
                                  path=f"{OUTPUT_DIR}/{spec.key}/figs/fig7_head_grid_L{before}.pdf")
        plt.show()
        print(f"Layer {before}: no sink column. Layer {FOCUS}: the stripe is there. "
              f"Sinks switch on at layer {first_sink}.")

---
## 7. Shared direction: identify v\*, the direction the register tokens share

The claim being tested has four separable parts, and each gets its own number rather than one
impression:

1. **Registers point one way.** v\* is the top right-singular vector of the matrix of *unit*
   register directions, and we report the share of variance it explains. SVD is the right tool
   here because it is sign-free: a population split into `+v` and `−v` still resolves to one axis
   instead of averaging to zero, unlike sign-aligning to a reference vector and taking a mean,
   which silently *assumes* the answer.
2. **It does not depend on the prompt or the seed.** v\* is re-fitted independently per
   (prompt, seed) and the pairwise |cos| between those fits is reported.
3. **One channel carries most of it.** Energy share of v\*'s top channel, and whether that channel
   is the model's loudest massive-activation channel.
4. **Selection is by direction, not magnitude.** Rank every token by `|x·v*|` and by `‖x‖`, and ask
   which ranking's top 1% contains the layer's sinks.

The matched null throughout is **ordinary tokens from the same layers**, not random vectors: the
question is whether registers are special, not whether they beat noise.


In [ ]:
# 7.1: Fit v* and read the numbers.
report = fit_vstar(result)
print(report.summary())

In [ ]:
# 7.2: The identity card for v*.
F.fig_vstar_identity(report, result, show_chance=SHOW_CHANCE_BASELINE, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig9_vstar_identity.pdf")
plt.show()

print("v* top channels by energy:")
display(pd.DataFrame({
    "rank": range(1, len(report.top_channels) + 1),
    "channel": report.top_channels,
    "energy_share": [f"{e:.1%}" for e in report.top_channel_energy],
    "cumulative": [f"{c:.1%}" for c in report.cumulative_energy[:len(report.top_channels)]],
}))

In [ ]:
# 7.3: The channel bridge: are the register tokens the spike positions of the massive
#       channel, and are those same tokens the attention sinks?
F.fig_vstar_channel_bridge(report, result, layer=FOCUS,
                           path=f"{OUTPUT_DIR}/{spec.key}/figs/fig10_channel_bridge.pdf")
plt.show()

In [ ]:
# 7.4a: The channel landscape: how far the massive channels stand above the bulk.
F.fig_channel_landscape(result, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig16_channel_landscape.pdf")
plt.show()

In [ ]:
# 7.4b: Where the register direction is written into the residual stream.
F.fig_vstar_projection_trajectory(report, result,
                                  path=f"{OUTPUT_DIR}/{spec.key}/figs/fig17_vstar_projection.pdf")
plt.show()

In [ ]:
# 7.4: Exact direction-vs-magnitude test (optional; one extra pass over the generations).
# Panel (e) above uses projections onto the stored loud channels only, which covers
# report.coverage of v*'s energy. This pass computes x . v* exactly at every layer.
RUN_EXACT_PROJECTION = False

if RUN_EXACT_PROJECTION:
    from ditsinks.runner import run_projection_sweep
    proj = run_projection_sweep(cfg, report.v)
    report = attach_exact_selection(report, result, proj["rows"])
    display(report.selection.round(3))
    F.fig_vstar_identity(report, result,
                         path=f"{OUTPUT_DIR}/{spec.key}/figs/fig9_vstar_identity_exact.pdf")
    plt.show()
else:
    print(f"approximate: panel (e) covers {report.coverage:.0%} of v*'s energy "
          f"(the {cfg.channel_topk} loudest channels per layer). "
          "Set RUN_EXACT_PROJECTION = True for the exact version.")

In [ ]:
# 7.5: Save v* so a later notebook or intervention can transplant, ablate or rotate it.
import torch as _torch
vstar_path = Path(OUTPUT_DIR) / spec.key / "vstar.pt"
vstar_path.parent.mkdir(parents=True, exist_ok=True)
_torch.save({
    "v": report.v,
    "model": spec.repo_id,
    "explained_variance": report.explained_variance,
    "top_channels": report.top_channels,
    "top_channel_energy": report.top_channel_energy,
    "massive_channel": report.massive_channel,
    "layers_used": report.layers_used,
    "config": cfg.to_json(),
}, vstar_path)
print("saved", vstar_path)

### 7.6 Why do image queries prefer natural register keys?

This is an observational query-key check. The capture stores only
the cosine between each head's mean image query and every natural image key after QK normalization
and rotary embedding. It does not store full Q/K tensors and does not claim that $W_Kv^*$ is
sufficient. Saved sweeps created before this cell was added must be regenerated once.


In [ ]:
# 7.6: Natural register-key rank and query-key alignment.
try:
    F.fig_qk_geometry(
        result, path=f"{OUTPUT_DIR}/{spec.key}/figs/paper_fig4_qk_geometry.pdf")
    plt.show()
except ValueError as e:
    print("QK geometry unavailable:", e)
    print("Re-run §4 once with the updated capture code; no full Q/K matrices are retained.")


### 7.7 The sparse signatures have a lifecycle

The four rows share a layer axis but retain their own units. Co-emergence is descriptive. The
direction-versus-magnitude interventions in §9 assign distinct causal roles.


In [ ]:
# 7.7: Dominant-channel, shared-axis, norm, and sink signatures across depth.
F.fig_sparse_lifecycle(
    result, report, path=f"{OUTPUT_DIR}/{spec.key}/figs/paper_fig5d_sparse_lifecycle.pdf")
plt.show()


---
## 8. Coupling and robustness

Section 5 establishes *where* each phenomenon lives. This section asks whether the high-norm
tokens and the sinks are the **same tokens**, and whether that answer survives a change in how
"high-norm" is defined.

### The random-overlap baseline (“Chance”)

“Chance” does **not** mean a statistical p-value and it is not a claim that the model is random.
It is the overlap expected if two sets with the observed sizes were placed independently among
$N$ tokens. For two top-$k$ sets, the expected intersection is $k^2/N$; the plotted Jaccard
baseline uses that expected intersection divided by the corresponding expected union. A measured
overlap above this grey reference says that high-norm tokens and sinks co-locate more often than
set size alone predicts. Near the line means the set overlap supplies little evidence of coupling.
The baseline is shown by default and can be hidden everywhere controlled
below by setting `SHOW_CHANCE_BASELINE = False` in §2.


In [ ]:
# 8.1: Set overlap at each residual-stream stage. Where the high-norm set is measured
#       matters: a token can be marked by attention and only amplified by the MLP after it.
F.fig_overlap_by_stage(result, show_chance=SHOW_CHANCE_BASELINE, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig18_overlap_by_stage.pdf")
plt.show()

In [ ]:
# 8.2: Rank correlation between norm and incoming attention, per head and layer.
F.fig_spearman_heatmap(result, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig19_rank_correlation.pdf")
plt.show()

In [ ]:
# 8.3: Two head-level views of the same coupling.
F.fig_sink_rank_histogram(result, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig20_sink_rank.pdf")
plt.show()
F.fig_head_sink_membership(result, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig21_head_membership.pdf")
plt.show()

### 8.4 Is any of this an artefact of the threshold?

An empirical percentile selects a fixed fraction of tokens *by construction*, so counting how many
tokens clear it proves nothing. Two checks: sweep the percentile and see whether the coupling
survives, and compare the threshold-to-median norm ratio against what a Gaussian residual would
give. If the ratio matches the Gaussian value there is no outlier population at that layer,
however many tokens the rule selects.

In [ ]:
# 8.4: Threshold sensitivity and the outlier-population test.
F.fig_percentile_sensitivity(result, show_chance=SHOW_CHANCE_BASELINE, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig22_threshold_sweep.pdf")
plt.show()
F.fig_norm_ratio_substrate(result, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig23_outlier_test.pdf")
plt.show()

In [ ]:
# 8.5: Individual token trajectories, and which patches get selected in the first place.
F.fig_token_trajectories(result, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig24_trajectories.pdf")
plt.show()
F.fig_register_selection(result, show_chance=SHOW_CHANCE_BASELINE, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig25_selection.pdf")
plt.show()

In [ ]:
# 8.6: Per-head competition between the text keys and the image sink (FLUX only).
try:
    F.fig_text_attention_per_head(result,
                                  path=f"{OUTPUT_DIR}/{spec.key}/figs/fig26_text_per_head.pdf")
    plt.show()
except ValueError as e:
    print("skipped:", e)

---
## 9. Causal interventions

Sections 5 to 8 are correlational: the high-norm tokens and the attention sinks are the same
tokens. This section edits the residual stream during generation and asks what the sinks do in
response, which is what separates *the sinks are the loud tokens* from *the sinks are the tokens
pointing along* $v^*$.

A small set of image tokens is edited at the input of one layer, at **every** denoising step, so
everything downstream sees the edited trajectory.

| Condition | What it does | What it decides |
|---|---|---|
| Baseline | nothing | reference |
| Instrumentation control | selects tokens, changes nothing | whether observation alone perturbs the model |
| Magnitude ablation | rescales targets to the median norm, direction untouched | whether the sink follows the norm |
| Direction ablation | replaces the direction, norm untouched | whether the sink follows the direction |
| Direction and magnitude ablation | replaces both | whether the two act together |
| Direction transfer | copies a target direction onto an ordinary token at that token's own norm | whether the direction alone is sufficient |
| Direction transfer at *k*x median norm | the same at a range of insertion norms | whether the direction needs magnitude to survive |

Two controls run alongside every ablation, editing the **same number of tokens at the same steps**:
a *matched-norm control* (the next-highest-norm tokens, which are not registers) and a
*random-token control*. Without them a change in the image cannot be attributed to the registers
rather than to editing anything at all.

Cost: one generation per condition per (prompt, seed). The ablation suite is 10 conditions, the
magnitude sweep adds 5.

In [ ]:
# 9.1: Configure. The edit lands at the input of one layer; by default the first
#       layer after high-norm tokens appear, which is the dual->single boundary in FLUX.
from ditsinks import interventions as IV

int_cfg = IV.InterventionConfig(
    base=cfg,
    intervention_layer=IV.suggest_intervention_layer(result),
    observe_layers=None,        # None -> the edit layer and a spread of layers after it
    target_ratio=cfg.highnorm_ratio,
    max_targets=8,
    conditions=IV.ABLATION_SUITE,
)
print(f"editing the input of layer {int_cfg.intervention_layer}")
print(f"conditions ({len(int_cfg.conditions)}):")
for k in int_cfg.conditions:
    print(f"  {IV.condition_label(k)}")
print(f"\n{len(int_cfg.conditions) * len(cfg.prompts) * len(cfg.seeds)} generations""")

In [ ]:
# 9.2: Run the ablation suite. Resume-safe: finished conditions are skipped.
suite = IV.run_intervention_suite(int_cfg)
iv = IV.load_intervention_results(suite["root"])
print(f"\nconditions on disk: {len(set(iv['head']['condition']))}")

In [ ]:
# 9.3: Validity check before anything else: does observation alone move the sinks?
key = ["prompt_id", "seed", "step", "layer", "head_id"]
h = iv["head"]
base = h[h.condition == "baseline"].set_index(key)["sink_token"]
sham = h[h.condition == "instrumentation_control"].set_index(key)["sink_token"]
agreement = (base.reindex(sham.index) == sham).mean()
print(f"instrumentation control reproduces the baseline sink in {agreement:.1%} of heads")
if agreement < 0.95:
    print("WARNING: the instrumentation itself is perturbing the model; "
          "no other condition below can be trusted until this is resolved.")

# Second check: were any tokens actually selected? If no token clears the
# target ratio, every ablation is a no-op and the section says nothing.
edited = iv["log"].groupby("condition")["n_edited"].mean()
print(f"\ntokens edited per step, by condition:")
for k, v in edited.items():
    print(f"  {IV.condition_label(k):48s} {v:.2f}")
if edited.get("magnitude_ablation", 0) == 0:
    print(f"\nWARNING: no token exceeds {int_cfg.target_ratio:g}x the median norm at layer "
          f"{int_cfg.intervention_layer}, so nothing was edited. Either this model has no "
          "registers at that layer, or target_ratio is too strict -- check section 5 first.")

### 9.4 Does the sink follow the norm, or the direction?

In [ ]:
# 9.4: The central result.
F.fig_sink_persistence(iv, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig27_sink_persistence.pdf")
plt.show()
display(IV.sink_persistence(iv).round(3))

In [ ]:
# 9.5: Consequences for the block: head consensus, the outlier norm downstream,
#       and whether attention moves onto the text keys instead.
F.fig_head_agreement(iv, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig28_head_agreement.pdf")
plt.show()
F.fig_norm_recovery(iv, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig29_norm_recovery.pdf")
plt.show()
try:
    F.fig_attention_reallocation(iv, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig30_reallocation.pdf")
    plt.show()
except ValueError as e:
    print("skipped:", e)

### 9.6 Does any of it change the image?

The registers are usually claimed to be functionally inert. The test is not whether the image
changes at all, but whether it changes **more than the matched-norm and random-token controls**,
which edit the same number of tokens at the same steps.

In [ ]:
# 9.6: Perceptual change, with paired tests against the controls.
#       Uses LPIPS if installed, otherwise RMSE.
dist = IV.image_distances(suite["root"], cfg)
tests = IV.paired_comparisons(dist)
if dist.empty:
    print("no generated images on disk for these conditions "
          "(expected for a synthetic run; a real run saves them)")
else:
    F.fig_image_change(dist, tests, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig31_image_change.pdf")
    plt.show()
    if not tests.empty:
        display(tests.round(4))

In [ ]:
# 9.7: The images themselves, and where the change lands.
if dist.empty:
    print("no images to show")
else:
    F.fig_image_grid(suite["root"], cfg, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig32_image_grid.pdf")
    plt.show()
    F.fig_image_difference(suite["root"], cfg,
                           path=f"{OUTPUT_DIR}/{spec.key}/figs/fig33_image_difference.pdf")
    plt.show()

### 9.8 Is the direction sufficient on its own?

The transfer copies a register's **unit** direction onto an ordinary token at that token's own
norm, so no magnitude is injected. If heads then sink on the recipient, direction is sufficient.
The magnitude sweep repeats the transfer at a range of insertion norms: attention normalises its
inputs, so capture at the edit layer should not depend on magnitude, and any rise at later layers
says the direction needs magnitude to *survive* the intervening blocks rather than to be selected.

In [ ]:
# 9.8: Sufficiency of the direction.
try:
    F.fig_transfer_capture(iv, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig34_transfer_capture.pdf")
    plt.show()
    display(IV.transfer_capture(iv).round(3))
except ValueError as e:
    print("skipped:", e, "- no register token was available to copy a direction from.")

In [ ]:
# 9.9: The magnitude sweep (5 more conditions). Set to True to run it.
RUN_MAGNITUDE_SWEEP = False

if RUN_MAGNITUDE_SWEEP:
    sweep_cfg = IV.InterventionConfig(
        base=cfg, intervention_layer=int_cfg.intervention_layer,
        observe_layers=int_cfg.observe_layers, target_ratio=int_cfg.target_ratio,
        max_targets=int_cfg.max_targets, conditions=IV.MAGNITUDE_SWEEP_SUITE)
    IV.run_intervention_suite(sweep_cfg)
    iv = IV.load_intervention_results(sweep_cfg.root)
    F.fig_transfer_magnitude(iv, path=f"{OUTPUT_DIR}/{spec.key}/figs/fig35_transfer_magnitude.pdf")
    plt.show()
else:
    print(f"skipped; {len(IV.MAGNITUDE_SWEEP_SUITE) * len(cfg.prompts) * len(cfg.seeds)} "
          "more generations")

---
## 10. The same analysis on three architectures

The point of the adapters: change one string and the whole notebook runs on a different family.
This section runs a *small* sweep (one prompt, one seed) on several models and puts the atlases
side by side, which is where the interesting contrast lives:

* FLUX's image queries share the sequence with 512 text keys, so an image sink competes with the
  text/padding sink; PixArt's image self-attention has no text keys at all, so its sinks are
  uncontaminated.
* FLUX.2's single blocks are *parallel* blocks: attention and MLP share one output projection,
  so there is no separate "post-attention residual" stage to attribute a register to.


In [ ]:
# 10.1: Small sweeps across families. Comment out anything you cannot download or fit.
COMPARE_MODELS = [
    "flux1-schnell",
    # "flux2-klein",       # ~9B : the fast open FLUX.2 model (what "flux2-schnell" means)
    # "flux2-dev",         # ~32B: needs quantize_4bit=True
    # "pixart-sigma-512",
]

comparison = {}
for name in COMPARE_MODELS:
    c = SweepConfig(model=name, prompts=PROMPTS[:1], seeds=SEEDS[:1],
                    height=SIZE, width=SIZE, output_dir=OUTPUT_DIR,
                    quantize_4bit=(name == "flux2-dev"))
    print(f"\n=== {name} ===")
    r = run_sweep(c)
    r.save()
    comparison[name] = r
    print(M.answer_where_are_the_sinks(r))

In [ ]:
# 10.2: Small multiples: rows = phenomenon, columns = model, x = relative depth.
if len(comparison) > 1:
    F.fig_model_comparison(comparison, path=f"{OUTPUT_DIR}/fig11_comparison.pdf")
    plt.show()

rows = []
for name, r in comparison.items():
    v = M.sink_layer_verdict(M.layer_table(r), threshold=r.cfg.sink_ratio_threshold)
    sinks = v[v["has_sinks"]]["layer"]
    hn = v[v["has_highnorm"]]["layer"]
    rows.append(dict(
        model=name, family=r.meta["family"], layers=r.meta["n_layers"],
        first_sink_layer=int(sinks.min()) if len(sinks) else None,
        sink_layers=int(len(sinks)),
        first_highnorm_layer=int(hn.min()) if len(hn) else None,
        peak_sink_ratio=round(float(v["sink_ratio"].max()), 1),
        peak_norm_ratio=round(float(v["norm_ratio"].max()), 1),
    ))
display(pd.DataFrame(rows))

---
## Paper-figure audit

This audit prevents a figure proposed in `PAPER_README.md` from silently being absent. A main
figure can contain several PDF panels in LaTeX, so the manifest lists every required generated
asset and whether its prerequisite experiment has run.


In [ ]:
# Generate or check every figure panel.
paper_dir = Path(OUTPUT_DIR) / spec.key / "paper_figures"
paper_dir.mkdir(parents=True, exist_ok=True)

paper_status = []
def paper_panel(figure, panel, build):
    path = paper_dir / f"{figure}_{panel}.pdf"
    try:
        fig = build(path)
        plt.show()
        paper_status.append(dict(figure=figure, panel=panel, status="generated", path=str(path)))
    except (ValueError, KeyError, FileNotFoundError, NameError) as e:
        paper_status.append(dict(figure=figure, panel=panel, status="missing prerequisite", path=str(path), reason=str(e)))

paper_panel("fig1", "depth_atlas", lambda p: F.fig_layer_atlas(result, path=p))
paper_panel("fig1", "spatial_example", lambda p: F.fig_spatial_token_maps(result, FOCUS, prompt_id=0, path=p))
paper_panel("fig2", "shared_axis", lambda p: F.fig_vstar_identity(report, result, show_chance=SHOW_CHANCE_BASELINE, path=p))
paper_panel("fig2", "channel_bridge", lambda p: F.fig_vstar_channel_bridge(report, result, layer=FOCUS, path=p))
paper_status.append(dict(figure="fig2", panel="controlled_channel_ablation", status="missing prerequisite", reason="import same-checkpoint dominant, matched-massive, random, and ideally norm-restored channel-ablation results"))
paper_panel("fig4", "qk_geometry", lambda p: F.fig_qk_geometry(result, path=p))
paper_panel("fig5", "lifecycle", lambda p: F.fig_sparse_lifecycle(result, report, path=p))

if "iv" in globals():
    paper_panel("fig3", "direction_vs_magnitude", lambda p: F.fig_sink_persistence(iv, path=p))
else:
    paper_status.append(dict(figure="fig3", panel="direction_vs_magnitude", status="missing prerequisite", reason="run §9 intervention suite"))
paper_status.append(dict(figure="fig3", panel="generation_level_intervals", status="missing prerequisite", reason="add prompt/seed-level paired intervals before paper freeze"))
if "iv" in globals():
    paper_panel("fig5", "magnitude_persistence", lambda p: F.fig_transfer_magnitude(iv, path=p))
else:
    paper_status.append(dict(figure="fig5", panel="magnitude_persistence", status="missing prerequisite", reason="run §9.9 magnitude sweep"))

paper_status = pd.DataFrame(paper_status)
display(paper_status)
paper_status.to_csv(paper_dir / "paper_figure_manifest.csv", index=False)
missing = paper_status[paper_status.status != "generated"]
if not missing.empty:
    print("\nMAIN-PAPER PANELS STILL MISSING:", missing.to_string(index=False), sep="\n")


---
## 11. Export

In [ ]:
# 11.1: Tables for the paper / for downstream analysis.
out = Path(OUTPUT_DIR) / spec.key / "tables"
out.mkdir(parents=True, exist_ok=True)

layer_df.to_csv(out / "layer_metrics.csv", index=False)
verdict.to_csv(out / "layer_verdict.csv", index=False)
M.head_table(result).to_csv(out / "head_metrics.csv", index=False)
report.layer_profile.to_csv(out / "vstar_layer_profile.csv", index=False)
report.selection.to_csv(out / "vstar_direction_vs_magnitude.csv", index=False)

for f in sorted(out.iterdir()):
    print(f.name, f"{f.stat().st_size/1e6:.2f} MB")
print("\nfigures ->", Path(OUTPUT_DIR) / spec.key / "figs")

In [ ]:
# 11.2: Suggested LaTeX captions.
# Every figure carries one in `fig.dv_caption`, written to drop straight into \\caption{...}.
caption_figs = {
    "fig2_layer_atlas": lambda: F.fig_layer_atlas(result),
    "fig3_depth_profile": lambda: F.fig_layer_summary(result),
    "fig5_layer_head_sinks": lambda: F.fig_layer_head_sinks(result),
    "fig7_head_grid": lambda: F.fig_attention_head_grid(result, FOCUS, prompt_id=0),
    "fig8_sink_profile": lambda: F.fig_sink_profile(result, FOCUS, prompt_id=0),
    "fig9_vstar_identity": lambda: F.fig_vstar_identity(report, result),
}
lines = []
for name, build in caption_figs.items():
    try:
        fig = build()
        lines.append(f"% {name}\n\\caption{{{fig.dv_caption}}}\n")
        plt.close(fig)
    except Exception as e:
        lines.append(f"% {name}: skipped ({e})\n")
print("\n".join(lines))
(Path(OUTPUT_DIR) / spec.key / "captions.tex").write_text("\n".join(lines))

---
## 12. Appendix: definitions, method, and what would falsify this

### Definitions used above

| term | definition | knob |
|---|---|---|
| high-norm token | image token whose `post_block` residual norm > `ratio` × that layer's median | `cfg.highnorm_ratio` (3.0) |
| attention sink | image key whose head-mean incoming image→image attention > `threshold` × the uniform share `1/N` | `cfg.sink_ratio_threshold` (10) |
| massive activation channel | channel whose peak \|activation\| over tokens > `ratio` × the median channel peak | `cfg.massive_channel_ratio` (10) |
| register | a high-norm token; its full hidden vector is what v\* is fitted on | `cfg.register_topk` (8/layer) |

The thresholds are all "× the layer's own typical value", not absolute. A fixed threshold would
select nothing early and everything late, because the bulk norm grows with depth.

A percentile-based definition is a different knob: an empirical percentile selects a fixed
*fraction* of tokens by construction, so counts alone can never demonstrate outliers. The
ratio-to-median form above does not have that problem; `M.layer_table(result, ratio=...)`
re-runs everything at a different threshold if you want the sensitivity curve.

### Method notes

* **The model's own attention runs.** `AttentionTap` rebinds the single function each family
  funnels its attention through (`dispatch_attention_fn` for FLUX, `F.scaled_dot_product_attention`
  for the PixArt/`Attention` path), calls the real kernel, and computes statistics from the same
  `q, k` afterwards in fp32. §3.1 asserts the output is bit-identical.
* **Image tokens are the tail.** FLUX.1, FLUX.2 and PixArt all put text before image when the two
  share a sequence, so "image tokens = the last `n_img` positions" holds across families *and*
  across the diffusers versions that moved FLUX.1's single blocks from a concatenated stream to a
  split pair.
* **Classifier-free guidance.** PixArt duplicates the batch as `[uncond, cond]`, and FLUX.2-klein
  runs two separate transformer calls per step. Both are handled: the analysis reads the
  conditional row and the conditional pass.
* **Attention scale.** Every processor calls its attention kernel without a `scale` argument, so
  `1/sqrt(head_dim)` is used; the tap reads a `scale` kwarg if one ever appears.

### What would falsify the story

* **Sink layers.** If the sink strength curve never rises above ~1×, this model has no sinks; if it rises
  in the same layers for shuffled prompts *and* the sinks land on different token ids each time,
  the phenomenon is positional noise rather than a register.
* **Shared direction.** If the variance explained by v\* is near `1/d_model`, the registers do not share a
  direction; if the per-prompt/seed |cos| matrix is not close to 1, v\* is prompt-specific; if
  ordinary tokens have the same |cos| distribution as registers, v\* is an artefact of the fit.
* Everything here is **correlational**. Causal interventions (magnitude clamp, direction rotation,
  transplant, dose-response) are what turn "the sinks are the tokens pointing along v\*" into a
  causal claim, and they read `vstar.pt` from §7.5.
